### **Notebook 1 (etapa 3 y 4): Entrenamiento y evaluación de regresión logística**

In [ ]:
import pandas as pd  # Permite el manejo y análisis de estructuras de datos (DataFrames)
import numpy as np  # Facilita la realización de cálculos numéricos y el manejo de matrices
import os  # Interacción con el sistema operativo (creación y verificación de rutas/directorios)
import gc  # Recolección de basura (Garbage Collector) para liberar memoria RAM
import time  # Medición de los tiempos de ejecución de las tareas
import joblib  # Serialización y guardado de los modelos entrenados de Machine Learning en disco
from sklearn.model_selection import GridSearchCV  # Realiza búsquedas exhaustivas de hiperparámetros
from sklearn.model_selection import StratifiedKFold  # Divide los datos en pliegues preservando el porcentaje de muestras de cada clase
from sklearn.linear_model import LogisticRegression  # Algoritmo principal que se entrenará (Regresión Logística)
from sklearn.preprocessing import StandardScaler, label_binarize  # Estandariza variables numéricas y binariza etiquetas multiclase
from sklearn.pipeline import Pipeline  # Construye secuencias de transformaciones y entrenamiento
from sklearn.base import clone  # Permite clonar estimadores sin copiar los datos originales
from sklearn.metrics import (  # Colección de funciones para evaluar el rendimiento del modelo
    f1_score, 
    average_precision_score, 
    roc_auc_score, 
    brier_score_loss, 
    classification_report 
)

def entrenar_evaluar_lr(target_name):
    """
    Descripción:
        Entrena, optimiza mediante GridSearch, y evalúa un modelo de Regresión Logística usando un Pipeline.
        Se encarga de balancear los datos de entrada, buscar la mejor combinación de hiperparámetros que 
        ofrezca estabilidad en el tiempo, evaluar el conjunto de pruebas extrayendo métricas y Odds Ratios,
        y exportar todos los resultados y el modelo óptimo a disco.

    Entradas:
        - target_name (str): Nombre exacto de la columna que representa la variable objetivo (target) a predecir.

    Salidas:
        - None: La función no retorna elementos directamente en memoria, pero guarda en disco local:
            1. Un archivo CSV con los resultados de la validación cruzada (GridSearch).
            2. El modelo serializado (.pkl) con la mejor configuración de hiperparámetros.
            3. Un archivo CSV con el reporte de métricas desglosado por clases.
            4. Archivos CSV con los coeficientes originales y los Odds Ratios del modelo.
    """
    
    # 1. Configuración inicial
    # Definir el directorio de lectura de datos
    dir_datos = "../../Datos/Datasets Finales" 
    # Definir y crear el directorio para almacenar los resultados si no existe
    dir_resultados = "../../Resultados/Resultados (etapa 3 y 4)/Regresion_Logistica" 
    os.makedirs(dir_resultados, exist_ok=True) 

    # Lista de variables que no deben ser incluidas como features en el modelo
    cols_excluir = ['CONSUMO_RECURSOS', 'SEVERIDAD', 'MORTALIDAD', 'CATEGORIA_CANCER'] 

    print("="*60) 
    print(f"Iniciando entrenamiento y evaluación de Regresión Logística para la variable objetivo: {target_name.upper()}")
    print("="*60) 

    # 2. Cargar datos de entrenamiento
    print("[1/5] Cargando datasets de entrenamiento...") 
    # Lectura del dataset con casos positivos/oncológicos
    df_onco_train = pd.read_csv(os.path.join(dir_datos, "dataset_entrenamiento_onco.csv"), low_memory=False) 
    # Lectura del dataset con casos negativos/de control
    df_control_train = pd.read_csv(os.path.join(dir_datos, "dataset_entrenamiento_control.csv"), low_memory=False) 

    # 3. Crear Dataset Maestro Balanceado 
    print("[2/5] Generando muestra balanceada...") 
    # Obtener el número de registros oncológicos para equilibrar las clases
    n_onco = len(df_onco_train) 
    # Tomar una muestra aleatoria (con semilla fija) de casos de control equivalente al tamaño oncológico
    df_control_sample = df_control_train.sample(n=n_onco, random_state=42) 
    # Unificar los subconjuntos para crear el dataframe de entrenamiento final
    df_train_maestro = pd.concat([df_onco_train, df_control_sample], ignore_index=True) 

    # Eliminar dataframes intermedios y llamar al colector de basura para liberar memoria
    del df_onco_train, df_control_train, df_control_sample 
    gc.collect() 

    # Filtrar las columnas para dejar únicamente las características (features)
    features = [col for col in df_train_maestro.columns if col not in cols_excluir] 
    # Asignar features a X_train y target a y_train
    X_train = df_train_maestro[features] 
    y_train = df_train_maestro[target_name] 
    
    # Identificar la cantidad de clases presentes para saber si es clasificación binaria o multiclase
    clases_unicas = np.unique(y_train) 
    is_multiclass = len(clases_unicas) > 2 

    print(f"      -> Dimensiones: {X_train.shape} | Clases: {len(clases_unicas)} (Multiclase: {is_multiclass})") 

    # 4. Configurar Grid Search (5 Pliegues) CON PIPELINE Y SEMILLA
    print("[3/5] Configurando Grid Search CV (K=5) con Pipeline y semilla...") 
    # Establecer la validación cruzada estratificada para preservar la proporción de clases en 5 cortes
    cv_estrategia = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    # Determinar la estrategia multiclase apropiada para la regresión logística
    multi_class_strat = 'multinomial' if is_multiclass else 'auto' 

    # Construir un pipeline que primero escala las variables (Z-score) y luego aplica el modelo
    pipeline = Pipeline([ 
        ('scaler', StandardScaler()), 
        ('lr', LogisticRegression(solver='saga', max_iter=1000, class_weight='balanced',  
                                  multi_class=multi_class_strat, random_state=42, n_jobs=1)) 
    ])

    # Definir la grilla de hiperparámetros a explorar (tipo de penalización y nivel de regularización)
    espacio_hiperparametros = { 
        'lr__penalty': ['l1', 'l2'], 
        'lr__C': [0.1, 1.0, 10.0] 
    }

    # Configurar el buscador exhaustivo con validación cruzada optimizando el 'f1_macro'
    grid_search = GridSearchCV( 
        estimator=pipeline, 
        param_grid=espacio_hiperparametros, 
        cv=cv_estrategia, 
        scoring='f1_macro', 
        n_jobs=-1,  # Utilizar todos los núcleos disponibles de la CPU
        verbose=3 
    )

    # 5. Entrenar y extraer métricas filtradas
    print("[4/5] Entrenando modelo y evaluando configuraciones...") 
    # Marcar el tiempo de inicio
    inicio = time.time() 
    # Ejecutar la búsqueda de parámetros sobre los datos de entrenamiento
    grid_search.fit(X_train, y_train) 
    # Marcar tiempo de fin y calcular duración
    fin = time.time() 
    print(f"      -> Búsqueda completada en {round((fin - inicio)/60, 2)} minutos.") 

    # Transformar el resumen del GridSearch a DataFrame y exportar
    cv_resultados = pd.DataFrame(grid_search.cv_results_) 
    ruta_cv = os.path.join(dir_resultados, f"Resultados_GridSearch_LR_{target_name}.csv") 
    cv_resultados.to_csv(ruta_cv, index=False) 
    print(f"      -> Evidencia de hiperparámetros guardada en: {ruta_cv}") 

    # Filtrar solo las configuraciones que tienen una desviación estándar pequeña en los pliegues (estables)
    config_estables = cv_resultados[cv_resultados['std_test_score'] <= 0.10] 

    if config_estables.empty: 
        print("      ADVERTENCIA: Todas las configuraciones tienen std > 0.10.") 
        print("      Se utilizará la de mayor promedio por defecto.") 
        # Si no hay configuraciones estables, usar el modelo ganador estándar
        mejor_modelo = grid_search.best_estimator_ 
    else: 
        # Obtener el índice del hiperparámetro que ofrece el mayor F1 Score promedio entre los estables
        mejor_indice = config_estables['mean_test_score'].idxmax() 
        mejores_params = config_estables.loc[mejor_indice, 'params'] 
        mejor_f1 = config_estables.loc[mejor_indice, 'mean_test_score'] 
        mejor_std = config_estables.loc[mejor_indice, 'std_test_score'] 
        
        print(f"      -> Mejor configuración estable encontrada:") 
        print(f"         Hiperparámetros: {mejores_params}") 
        print(f"         F1-Macro Promedio: {mejor_f1:.4f} (std: {mejor_std:.4f})") 

        # Clonar el Pipeline inicial, aplicar los mejores hiperparámetros, y volver a entrenar toda la data
        mejor_modelo = clone(grid_search.estimator) 
        mejor_modelo.set_params(**mejores_params) 
        mejor_modelo.fit(X_train, y_train) 

    # --- GUARDADO DEL MODELO MAESTRO EN DISCO ---
    # Configurar ruta del archivo pickle para el modelo final
    ruta_modelo = os.path.join(dir_resultados, f"Modelo_Optimo_RL_{target_name}.pkl")
    # Exportar el pipeline completo usando joblib
    joblib.dump(mejor_modelo, ruta_modelo)
    print(f"      -> Modelo óptimo guardado en: {ruta_modelo}")
    # --------------------------------------------

    # --- MÉTRICAS DE ENTRENAMIENTO ---
    print("\n--- Rendimiento en entrenamiento: ---")
    # Realizar predicciones sobre el conjunto con el que se entrenó
    y_pred_train = mejor_modelo.predict(X_train)
    # Medir e imprimir métricas sobre la fase de entrenamiento
    if is_multiclass:
        print(f"F1-Score (Macro) Train: {f1_score(y_train, y_pred_train, average='macro'):.4f}")
    else:
        print(f"F1-Score (Clase 1) Train: {f1_score(y_train, y_pred_train, pos_label=1):.4f}")
    # ---------------------------------

    # Liberar memoria de datos de entrenamiento
    del df_train_maestro, X_train, y_train 
    gc.collect() 

    # 6. Evaluación en el Conjunto de Prueba
    print("[5/5] Evaluando en conjunto de prueba ...") 
    # Cargar los datasets de test que no fueron vistos durante el entrenamiento
    df_onco_test = pd.read_csv(os.path.join(dir_datos, "dataset_prueba_onco.csv"), low_memory=False)
    df_control_test = pd.read_csv(os.path.join(dir_datos, "dataset_prueba_control.csv"), low_memory=False) 

    # Unir ambas tablas y separar features/target
    df_test_maestro = pd.concat([df_onco_test, df_control_test], ignore_index=True) 
    X_test = df_test_maestro[features] 
    y_test = df_test_maestro[target_name] 
    total_instancias = len(y_test) 

    # Calcular las predicciones duras y las probabilidades para el set de prueba
    y_pred = mejor_modelo.predict(X_test) 
    y_pred_proba = mejor_modelo.predict_proba(X_test) 

    print("\n--- Resultados finales en evaluación ---") 
    # Mostrar por consola el reporte general de clasificación
    print(classification_report(y_test, y_pred)) 
    
    # --- EXPORTAR REPORTE A CSV ---
    # Transformar el reporte de sklearn en diccionario y luego en dataframe para exportar
    reporte_dic = classification_report(y_test, y_pred, output_dict=True)
    df_reporte = pd.DataFrame(reporte_dic).transpose()
    ruta_reporte = os.path.join(dir_resultados, f"Reporte_Desglose_Clases_LR_{target_name}.csv")
    df_reporte.to_csv(ruta_reporte)
    print(f"      -> Reporte de desglose por clases guardado en: {ruta_reporte}")
    # ------------------------------
    
    # Obtener métricas generales F1-Macro
    f1_macro_val = f1_score(y_test, y_pred, average='macro') 
    
    # Evaluar métricas específicas según la naturaleza de la clasificación (Multiclase o Binaria)
    if is_multiclass: 
        # Binarizar el conjunto de prueba para que funcionen métricas especializadas One-vs-Rest (OvR)
        y_test_bin = label_binarize(y_test, classes=clases_unicas) 
        auc_roc_val = roc_auc_score(y_test, y_pred_proba, multi_class='ovr', average='weighted') 
        auprc_val = average_precision_score(y_test_bin, y_pred_proba, average='weighted') 
        
        # Calcular la métrica Brier score promediando los errores de las probabilidades en cada clase
        brier_val = np.mean([brier_score_loss(y_test_bin[:, k], y_pred_proba[:, k]) for k in range(len(clases_unicas))])
        
        # Calcular tasa base ponderada analizando las frecuencias de las clases
        _, soportes_clases = np.unique(y_test, return_counts=True)
        prevalencias = [soporte / total_instancias for soporte in soportes_clases]
        tasa_base = sum([p**2 for p in prevalencias]) 

        # Imprimir resultados multiclase
        print(f"F1-Score (Macro): {f1_macro_val:.4f}") 
        print(f"AUPRC (OvR Weighted): {auprc_val:.4f}") 
        print(f"AUC-ROC (OvR Weighted): {auc_roc_val:.4f}") 
        print(f"Brier Score (Multiclase): {brier_val:.4f}") 
        
        # 7. Extraer Odds Ratios por clase
        # Rescatar las desviaciones estándar del escalador para calcular coeficientes en escala natural
        desviaciones_std = mejor_modelo.named_steps['scaler'].scale_ 
        for idx, clase in enumerate(clases_unicas): 
            # Calcular coeficiente original para la clase en iteración
            coef_lr = mejor_modelo.named_steps['lr'].coef_[idx] 
            coef_orig = coef_lr / desviaciones_std 
            
            # Generar tabla resumen y calcular el exponente de los coeficientes (Odds Ratios)
            df_coef = pd.DataFrame({ 
                'Variable': features, 
                'Coeficiente_Original': coef_orig, 
                'Odds_Ratio': np.exp(coef_orig) 
            }).sort_values(by='Odds_Ratio', ascending=False) 
            
            # Exportar la tabla de Odds Ratios por cada clase al disco local
            ruta_coef = os.path.join(dir_resultados, f"LR_OddsRatios_{target_name}_Clase_{clase}.csv") 
            df_coef.to_csv(ruta_coef, index=False) 
            
    else: 
        # Calcular métricas para una clasificación binaria clásica
        f1_clase1_val = f1_score(y_test, y_pred, pos_label=1) 
        auc_roc_val = roc_auc_score(y_test, y_pred_proba[:, 1]) 
        auprc_val = average_precision_score(y_test, y_pred_proba[:, 1]) 
        brier_val = brier_score_loss(y_test, y_pred_proba[:, 1]) 
        
        # Extraer la prevalencia (tasa base) específica de la clase positiva (1)
        clases_temp, soportes_clases = np.unique(y_test, return_counts=True)
        indice_clase_1 = np.where(clases_temp == 1)[0][0]
        tasa_base = soportes_clases[indice_clase_1] / total_instancias

        # Imprimir resultados binarios
        print(f"F1-Score (Clase 1): {f1_clase1_val:.4f}") 
        print(f"F1-Score (Macro): {f1_macro_val:.4f}") 
        print(f"AUPRC: {auprc_val:.4f}") 
        print(f"AUC-ROC: {auc_roc_val:.4f}") 
        print(f"Brier Score: {brier_val:.4f}") 

        # 7. Extraer Odds Ratios Binario
        # Reversar el escalado de características para obtener un modelo interpretable en unidades reales
        coef_lr = mejor_modelo.named_steps['lr'].coef_[0] 
        desviaciones_std = mejor_modelo.named_steps['scaler'].scale_ 
        coef_orig = coef_lr / desviaciones_std 
        
        # Organizar tabla de coeficientes y calcular Odds Ratios ordenándolos descendentemente
        df_coef = pd.DataFrame({ 
            'Variable': features, 
            'Coeficiente_Original': coef_orig, 
            'Odds_Ratio': np.exp(coef_orig) 
        }).sort_values(by='Odds_Ratio', ascending=False) 
        
        # Guardar tabla de impacto de variables en local
        ruta_coef = os.path.join(dir_resultados, f"LR_OddsRatios_{target_name}.csv") 
        df_coef.to_csv(ruta_coef, index=False)  

    # Analizar si el modelo ofrece una mejora significativa frente al puro azar (validación Lift)
    print("\n" + "-" * 60)
    print(f"Validación de Lift (en AUPRC): {target_name.upper()}")
    print("-" * 60)
    print(f"Total episodios de prueba: {total_instancias}")
    print(f"Tasa base (Prevalencia Azar): {tasa_base:.4f} ({tasa_base*100:.2f}%)")
    print(f"AUPRC obtenido por el modelo: {auprc_val:.4f}")
    
    # Calcular el nivel de lift real contra el modelo aleatorio base
    umbral_minimo = tasa_base * 3.0
    lift_real = auprc_val / tasa_base
    
    print(f"Lift real logrado: {lift_real:.2f}x")
    
    # Para datasets desbalanceados (<15% prevalencia), verificar si el AUPRC triplica el azar
    if tasa_base < 0.15: 
        print(f"AUPRC Mínimo exigido (Tasa Base x 3.0): {umbral_minimo:.4f}")
        if auprc_val > umbral_minimo:
            print("Resultado: Cumple condición de Lift > 3.0")
        else:
            print("Resultado: No cumple condición de Lift > 3.0")
    else:
        print("Resultado: Target suficientemente balanceado")

    print(f"Resultados y Odds Ratios guardados en: {dir_resultados}") 
    
    # Liberar la memoria final después de toda la ejecución
    del df_test_maestro, X_test, y_test 
    gc.collect() 
    print("="*60, "\n")

In [2]:
entrenar_evaluar_lr('MORTALIDAD')

Iniciando entrenamiento y evaluación de Regresión Logística para la variable objetivo: MORTALIDAD
[1/5] Cargando datasets de entrenamiento...
[2/5] Generando muestra balanceada...
      -> Dimensiones: (780416, 110) | Clases: 2 (Multiclase: False)
[3/5] Configurando Grid Search CV (K=5) con Pipeline y semilla...
[4/5] Entrenando modelo y evaluando configuraciones...
Fitting 5 folds for each of 6 candidates, totalling 30 fits


c:\ProgramData\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


      -> Búsqueda completada en 77.86 minutos.
      -> Evidencia de hiperparámetros guardada en: ../../Resultados/Resultados (etapa 3)/Regresion_Logistica\Resultados_GridSearch_LR_MORTALIDAD.csv
      -> Mejor configuración estable encontrada:
         Hiperparámetros: {'lr__C': 1.0, 'lr__penalty': 'l1'}
         F1-Macro Promedio: 0.6024 (std: 0.0009)


c:\ProgramData\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


      -> Modelo óptimo guardado en: ../../Resultados/Resultados (etapa 3)/Regresion_Logistica\Modelo_Optimo_RL_MORTALIDAD.pkl

--- Rendimiento en entrenamiento: ---
F1-Score (Clase 1) Train: 0.3061
[5/5] Evaluando en conjunto de prueba ...

--- Resultados finales en evaluación ---
              precision    recall  f1-score   support

           0       1.00      0.87      0.93   1111727
           1       0.17      0.86      0.28     33407

    accuracy                           0.87   1145134
   macro avg       0.58      0.86      0.60   1145134
weighted avg       0.97      0.87      0.91   1145134

      -> Reporte de desglose por clases guardado en: ../../Resultados/Resultados (etapa 3)/Regresion_Logistica\Reporte_Desglose_Clases_LR_MORTALIDAD.csv
F1-Score (Clase 1): 0.2772
F1-Score (Macro): 0.6027
AUPRC: 0.2968
AUC-ROC: 0.9375
Brier Score: 0.0887

------------------------------------------------------------
Validación de Lift (en AUPRC): MORTALIDAD
--------------------------------

In [3]:
entrenar_evaluar_lr('SEVERIDAD')

Iniciando entrenamiento y evaluación de Regresión Logística para la variable objetivo: SEVERIDAD
[1/5] Cargando datasets de entrenamiento...
[2/5] Generando muestra balanceada...
      -> Dimensiones: (780416, 110) | Clases: 4 (Multiclase: True)
[3/5] Configurando Grid Search CV (K=5) con Pipeline y semilla...
[4/5] Entrenando modelo y evaluando configuraciones...
Fitting 5 folds for each of 6 candidates, totalling 30 fits


c:\ProgramData\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\ProgramData\anaconda3\Lib\site-packages\sklearn\linear_model\_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


      -> Búsqueda completada en 218.61 minutos.
      -> Evidencia de hiperparámetros guardada en: ../../Resultados/Resultados (etapa 3)/Regresion_Logistica\Resultados_GridSearch_LR_SEVERIDAD.csv
      -> Mejor configuración estable encontrada:
         Hiperparámetros: {'lr__C': 10.0, 'lr__penalty': 'l1'}
         F1-Macro Promedio: 0.7131 (std: 0.0013)


c:\ProgramData\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\ProgramData\anaconda3\Lib\site-packages\sklearn\linear_model\_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


      -> Modelo óptimo guardado en: ../../Resultados/Resultados (etapa 3)/Regresion_Logistica\Modelo_Optimo_RL_SEVERIDAD.pkl

--- Rendimiento en entrenamiento: ---
F1-Score (Macro) Train: 0.7134
[5/5] Evaluando en conjunto de prueba ...

--- Resultados finales en evaluación ---
              precision    recall  f1-score   support

           0       0.87      0.97      0.92    186805
           1       0.73      0.81      0.77    475049
           2       0.51      0.35      0.41    264286
           3       0.64      0.68      0.66    218994

    accuracy                           0.70   1145134
   macro avg       0.69      0.70      0.69   1145134
weighted avg       0.69      0.70      0.69   1145134

      -> Reporte de desglose por clases guardado en: ../../Resultados/Resultados (etapa 3)/Regresion_Logistica\Reporte_Desglose_Clases_LR_SEVERIDAD.csv
F1-Score (Macro): 0.6893
AUPRC (OvR Weighted): 0.7555
AUC-ROC (OvR Weighted): 0.8841
Brier Score (Multiclase): 0.0979

---------------

In [4]:
entrenar_evaluar_lr('CONSUMO_RECURSOS')

Iniciando entrenamiento y evaluación de Regresión Logística para la variable objetivo: CONSUMO_RECURSOS
[1/5] Cargando datasets de entrenamiento...
[2/5] Generando muestra balanceada...
      -> Dimensiones: (780416, 110) | Clases: 3 (Multiclase: True)
[3/5] Configurando Grid Search CV (K=5) con Pipeline y semilla...
[4/5] Entrenando modelo y evaluando configuraciones...
Fitting 5 folds for each of 6 candidates, totalling 30 fits


c:\ProgramData\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\ProgramData\anaconda3\Lib\site-packages\sklearn\linear_model\_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


      -> Búsqueda completada en 119.3 minutos.
      -> Evidencia de hiperparámetros guardada en: ../../Resultados/Resultados (etapa 3)/Regresion_Logistica\Resultados_GridSearch_LR_CONSUMO_RECURSOS.csv
      -> Mejor configuración estable encontrada:
         Hiperparámetros: {'lr__C': 0.1, 'lr__penalty': 'l1'}
         F1-Macro Promedio: 0.6700 (std: 0.0005)


c:\ProgramData\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\ProgramData\anaconda3\Lib\site-packages\sklearn\linear_model\_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


      -> Modelo óptimo guardado en: ../../Resultados/Resultados (etapa 3)/Regresion_Logistica\Modelo_Optimo_RL_CONSUMO_RECURSOS.pkl

--- Rendimiento en entrenamiento: ---
F1-Score (Macro) Train: 0.6703
[5/5] Evaluando en conjunto de prueba ...

--- Resultados finales en evaluación ---
              precision    recall  f1-score   support

           0       0.67      0.84      0.75    384237
           1       0.58      0.57      0.57    380666
           2       0.77      0.58      0.66    380231

    accuracy                           0.67   1145134
   macro avg       0.67      0.67      0.66   1145134
weighted avg       0.67      0.67      0.66   1145134

      -> Reporte de desglose por clases guardado en: ../../Resultados/Resultados (etapa 3)/Regresion_Logistica\Reporte_Desglose_Clases_LR_CONSUMO_RECURSOS.csv
F1-Score (Macro): 0.6617
AUPRC (OvR Weighted): 0.7268
AUC-ROC (OvR Weighted): 0.8434
Brier Score (Multiclase): 0.1479

-------------------------------------------------------